# GED ML


In [1]:
import gc
import re
import resource
import warnings
from collections import Counter
from itertools import islice
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn_crfsuite
from camel_tools.disambig.mle import MLEDisambiguator
from camel_tools.utils.charsets import AR_CHARSET
from camel_tools.utils.normalize import normalize_alef_ar, normalize_alef_maksura_ar
from scipy.sparse import csr_matrix
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction import DictVectorizer, FeatureHasher
from sklearn.linear_model import SGDClassifier

warnings.filterwarnings("ignore", category=ConvergenceWarning)

## Data

loads the QALB14 splits


In [2]:
# ========================================
# DATA PATHS AND SHARED CONSTANTS
# ========================================

DATA_ROOT = Path("../../../../") / "src/services/ged/data/ml/qalb14/data"
TRAIN_PATH = DATA_ROOT / "train.txt"
DEV_PATH = DATA_ROOT / "dev.txt"
TEST_PATH = DATA_ROOT / "test.txt"

# NOTE: collected the labels in the QALB14 data and mapped them
# to our error taxonomy
LABEL_TO_CATEGORY = {
    "UC": "UC",
    "REPLACE_O": "OT",
    "REPLACE_P": "PC",
    "REPLACE_M": "MO",
    "REPLACE_S": "SY",
    "MERGE-B": "MG",
    "MERGE-I": "MG",
    "SPLIT": "SP",
    "DELETE": "UNK",
    "REPLACE_X": "UNK",
    "UNK": "UNK",
    "REPLACE_M+REPLACE_O": "MO",
    "REPLACE_O+REPLACE_X": "OT",
}
NO_ERROR = "UC"
# these are confidence thresholds for non UC labels
THRESHOLDS = np.arange(0.05, 0.91, 0.05)
PUNCT_RE = re.compile(r"^[^\w\s]+$", re.UNICODE)
N_FEATURES = 2**18


def read_split(path: Path):
    """Read a QALB split."""
    sentences = []
    sentence = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            if sentence:
                sentences.append(sentence)
                sentence = []
            continue
        token, label = line.split("\t")
        sentence.append((token, label))
    if sentence:
        sentences.append(sentence)
    return sentences


def audit_split(name: str, sentences: list[list[tuple[str, str]]]):
    """Summarize size and error ratio."""
    label_counts = Counter(label for sent in sentences for _, label in sent)
    token_count = sum(len(sent) for sent in sentences)
    error_count = token_count - label_counts["UC"]
    return {
        "split": name,
        "sentences": len(sentences),
        "tokens": token_count,
        "errors": error_count,
        "error_ratio": round(error_count / token_count, 4),
    }


# NOTE: I audit the splits first to make sure that train/dev/test match
train = read_split(TRAIN_PATH)
dev = read_split(DEV_PATH)
test = read_split(TEST_PATH)

pd.DataFrame(
    [
        audit_split("train", train),
        audit_split("dev", dev),
        audit_split("test", test),
    ]
)

,split,sentences,tokens,errors,error_ratio
0,train,19411,1021165,225655,0.2210
1,dev,1017,53737,11862,0.2207
2,test,968,51285,11595,0.2261


## Labels

maping the raw labels to Baligh categories


In [3]:
# ==============
# Label Mapping
# ==============


def sent_to_baligh_labels(sent: list[tuple[str, str]]):
    """Map QALB labels to Baligh categories."""
    return [LABEL_TO_CATEGORY[label] for _, label in sent]


def mapped_category_counts(sentences: list[list[tuple[str, str]]]):
    """Count Baligh categories."""
    return Counter(LABEL_TO_CATEGORY[label] for sent in sentences for _, label in sent)


category_counts = (
    pd.DataFrame(
        {
            "train": mapped_category_counts(train),
            "dev": mapped_category_counts(dev),
            "test": mapped_category_counts(test),
        }
    )
    .fillna(0)
    .astype(int)
)

category_counts

,train,dev,test
OT,142075,7404,6994
UC,795510,41875,39690
PC,11379,598,687
SY,5436,247,252
SP,7828,432,399
UNK,26869,1486,1583
MG,30359,1609,1602
MO,1709,86,78


## Expiriments


In [4]:
# ========
# HELPERS
# ========


def flatten(nested):
    """Flatten a nested list one level."""
    return [item for items in nested for item in items]


def binary_error_metrics(y_true: list[str], y_pred: list[str]):
    """Compute binary GED metrics where any non UC label counts as error."""
    true_error = np.asarray([label != NO_ERROR for label in y_true], dtype=np.bool_)
    pred_error = np.asarray([label != NO_ERROR for label in y_pred], dtype=np.bool_)

    tp = int(np.sum(true_error & pred_error))
    fp = int(np.sum(~true_error & pred_error))
    fn = int(np.sum(true_error & ~pred_error))
    tn = int(np.sum(~true_error & ~pred_error))

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "fp_per_1000": fp / len(y_true) * 1000,
    }


def category_detection_recall(y_true: list[str], y_pred: list[str]):
    """How often each true error category is detected as non-UC."""
    rows = {}
    for category in sorted(set(y_true) - {NO_ERROR}):
        indexes = [i for i, label in enumerate(y_true) if label == category]
        rows[category] = float(np.mean([y_pred[i] != NO_ERROR for i in indexes]))
    return rows


def normalize_token(token: str):
    """Apply light Arabic normalization."""
    return normalize_alef_maksura_ar(normalize_alef_ar(token))


def token_shape(token: str):
    """Convert a token into a digit/Arabic/other shape string ."""
    return "".join(
        "D" if char.isdigit() else "A" if char in AR_CHARSET else "P" for char in token
    )


def surface_v1_token_features(sent, index: int):
    """Extract surface_v1 features for one token."""
    token = sent[index][0]
    token_norm = normalize_token(token)

    row = {
        "bias": 1.0,
        "token": token,
        "norm": token_norm,
        "shape": token_shape(token),
        "len": len(token),
        "is_digit": token.isdigit(),
        "is_punct": bool(PUNCT_RE.match(token)),
        "is_arabic": bool(token) and all(char in AR_CHARSET for char in token),
    }

    for n in range(1, 5):
        row[f"prefix_{n}"] = token[:n]
        row[f"suffix_{n}"] = token[-n:]

    if index == 0:
        row["BOS"] = True
    else:
        previous = sent[index - 1][0]
        row["prev_token"] = previous
        row["prev_norm"] = normalize_token(previous)
        row["prev_is_punct"] = bool(PUNCT_RE.match(previous))

    if index == len(sent) - 1:
        row["EOS"] = True
    else:
        following = sent[index + 1][0]
        row["next_token"] = following
        row["next_norm"] = normalize_token(following)
        row["next_is_punct"] = bool(PUNCT_RE.match(following))

    return row


def sent_to_surface_v1_features(sent):
    """Extract surface_v1 features for asentence."""
    return [surface_v1_token_features(sent, i) for i in range(len(sent))]


def build_token_error_memory(train_sents, min_error_count: int = 2):
    """Memorize tokens that show  as errors often in training."""
    token_category_counts = {}
    for sent in train_sents:
        for token, label in sent:
            category = LABEL_TO_CATEGORY[label]
            if category == NO_ERROR or category == "UNK":
                continue
            token_category_counts.setdefault(token, Counter())
            token_category_counts[token][category] += 1

    memory = {}
    for token, counts in token_category_counts.items():
        category, count = counts.most_common(1)[0]
        if count >= min_error_count:
            memory[token] = {
                "category": category,
                "count": count,
                "category_counts": dict(counts),
            }
    return memory


def predict_with_token_memory(sentences, memory: dict):
    """See errors by looking up in tokens."""
    predictions = []
    for sent in sentences:
        sent_predictions = []
        for token, _ in sent:
            entry = memory.get(token)
            sent_predictions.append(entry["category"] if entry else NO_ERROR)
        predictions.append(sent_predictions)
    return predictions


def collect_token_error_stats(train_sents):
    """Token frequency and token error statistics once for sweeping thresholds."""
    token_stats = {}

    for sent in train_sents:
        for token, label in sent:
            category = LABEL_TO_CATEGORY[label]
            stats = token_stats.setdefault(
                token,
                {"total_count": 0, "error_count": 0, "category_counts": Counter()},
            )
            stats["total_count"] += 1

            if category != NO_ERROR and category != "UNK":
                stats["error_count"] += 1
                stats["category_counts"][category] += 1

    return token_stats


def build_calibrated_token_error_memory(
    token_stats: dict,
    min_total_count: int,
    min_error_count: int,
    min_error_rate: float,
) -> dict:
    """Build a token memory table using token error rate thresholds."""
    memory = {}
    for token, stats in token_stats.items():
        total_count = stats["total_count"]
        error_count = stats["error_count"]
        if total_count < min_total_count or error_count < min_error_count:
            continue
        error_rate = error_count / total_count
        if error_rate < min_error_rate:
            continue
        category, category_count = stats["category_counts"].most_common(1)[0]
        memory[token] = {
            "category": category,
            "total_count": total_count,
            "error_count": error_count,
            "error_rate": error_rate,
            "category_count": category_count,
        }
    return memory


def as_int32_csr(matrix) -> csr_matrix:
    """Cast sparse matrix index arrays to int32 for sklearn."""
    return csr_matrix(
        (
            matrix.data,
            matrix.indices.astype("int32"),
            matrix.indptr.astype("int32"),
        ),
        shape=matrix.shape,
    )


def predict_with_error_threshold(model, matrix, threshold: float):
    """Predict UC unless the best error class probability less than threshold."""
    probabilities = model.predict_proba(matrix)
    classes = list(model.classes_)
    error_indexes = [i for i, label in enumerate(classes) if label != NO_ERROR]
    predictions = []

    for row in probabilities:
        best_error_index = max(error_indexes, key=lambda i: row[i])
        if row[best_error_index] >= threshold:
            predictions.append(classes[best_error_index])
        else:
            predictions.append(NO_ERROR)

    return predictions


def predict_crf_with_error_threshold(model, X_sents, threshold: float):
    """Apply a probability threshold to CRF before predicting error."""
    marginal_sents = model.predict_marginals(X_sents)
    error_labels = [label for label in model.classes_ if label != NO_ERROR]
    predictions = []

    for sent_marginals in marginal_sents:
        sent_predictions = []

        for token_marginals in sent_marginals:
            best_error_label = max(
                error_labels,
                key=lambda label: token_marginals.get(label, 0.0),
            )

            if token_marginals.get(best_error_label, 0.0) >= threshold:
                sent_predictions.append(best_error_label)
            else:
                sent_predictions.append(NO_ERROR)

        predictions.append(sent_predictions)

    return predictions

In [5]:
# ====================
# E1: ALL-UC BASELINE
# ====================

y_train_baligh = [sent_to_baligh_labels(sent) for sent in train]
y_dev_baligh = [sent_to_baligh_labels(sent) for sent in dev]
flat_dev_baligh = flatten(y_dev_baligh)

# NOTE: This baseline predicts no errors at all.
all_uc_pred = [NO_ERROR] * len(flat_dev_baligh)
all_uc_row = {
    "experiment": "all_uc_baseline",
    **binary_error_metrics(flat_dev_baligh, all_uc_pred),
}

pd.DataFrame([all_uc_row])[["experiment", "f1", "precision", "recall", "fp_per_1000"]]

,experiment,f1,precision,recall,fp_per_1000
0,all_uc_baseline,0.0,0.0,0.0,0.0


In [6]:
# =================
# E2: TOKEN MEMORY
# =================

token_memory_rows = []
for threshold in [1, 2, 3, 5, 10]:
    memory = build_token_error_memory(train, min_error_count=threshold)
    predictions = flatten(predict_with_token_memory(dev, memory))
    token_memory_rows.append(
        {
            "experiment": "token_memory",
            "min_error_count": threshold,
            "memory_size": len(memory),
            **binary_error_metrics(flat_dev_baligh, predictions),
        }
    )

token_memory_best = max(token_memory_rows, key=lambda row: row["f1"])
token_memory_best["experiment"] = "token_memory_best_f1"

pd.DataFrame(token_memory_rows)

,experiment,min_error_count,memory_size,tp,fp,fn,tn,precision,recall,f1,fp_per_1000
0,token_memory,1,46916,9400,28240,2462,13635,0.249734,0.792446,0.379783,525.522452
1,token_memory,2,12767,8611,23257,3251,18618,0.270208,0.725932,0.393826,432.793048
2,token_memory,3,7271,8190,20662,3672,21213,0.283862,0.690440,0.402319,384.502298
3,token_memory,5,3838,7656,17755,4206,24120,0.301287,0.645422,0.410807,330.405493
4,token_memory_best_f1,10,1798,6975,14643,4887,27232,0.322648,0.588012,0.416667,272.493812


In [7]:
# ========================================
# E3: CALIBRATED TOKEN MEMORY
# ========================================

# NOTE: Here I keep the same idea as token memory, but I score tokens by how
# often they are wrong relative to how often they appear at all.
token_stats = collect_token_error_stats(train)
calibrated_rows = []
for min_error_rate in [0.05, 0.1, 0.2, 0.3, 0.4, 0.5]:
    memory = build_calibrated_token_error_memory(
        token_stats,
        min_total_count=1,
        min_error_count=1,
        min_error_rate=min_error_rate,
    )
    predictions = flatten(predict_with_token_memory(dev, memory))
    calibrated_rows.append(
        {
            "experiment": "calibrated_token_memory",
            "min_error_rate": min_error_rate,
            "memory_size": len(memory),
            **binary_error_metrics(flat_dev_baligh, predictions),
        }
    )

calibrated_memory_best = max(calibrated_rows, key=lambda row: row["f1"])
calibrated_memory_best["experiment"] = "calibrated_token_memory_best_f1"

pd.DataFrame(calibrated_rows)

,experiment,min_error_rate,memory_size,tp,fp,fn,tn,precision,recall,f1,fp_per_1000
0,calibrated_token_memory,0.05,45148,8840,8322,3022,33553,0.515091,0.745237,0.609151,154.865363
1,calibrated_token_memory,0.10,43867,8221,2662,3641,39213,0.755398,0.693053,0.722884,49.537563
2,calibrated_token_memory,0.20,42318,8057,1324,3805,40551,0.858864,0.679228,0.758556,24.638517
3,calibrated_token_memory,0.30,41140,7794,649,4068,41226,0.923132,0.657056,0.767693,12.077340
4,calibrated_token_memory_best_f1,0.40,40172,7687,461,4175,41414,0.943422,0.648036,0.768316,8.578819
5,calibrated_token_memory,0.50,40006,7638,391,4224,41484,0.951302,0.643905,0.767986,7.276178


In [8]:
# ============================
# E4: LINEAR MULTICLASS MODEL
# ============================

X_train_v1 = [sent_to_surface_v1_features(sent) for sent in train]
X_dev_v1 = [sent_to_surface_v1_features(sent) for sent in dev]
flat_X_train_v1 = flatten(X_train_v1)
flat_X_dev_v1 = flatten(X_dev_v1)
flat_y_train_baligh = flatten(y_train_baligh)

vectorizer = DictVectorizer(sparse=True)
linear_X_train = as_int32_csr(vectorizer.fit_transform(flat_X_train_v1))
linear_X_dev = as_int32_csr(vectorizer.transform(flat_X_dev_v1))

# NOTE: This is the main multiclass model in the notebook.
# It predicts Baligh categories directly afterthat i threshold non-UC confidence.
linear_model = SGDClassifier(loss="log_loss", random_state=7, max_iter=1000, tol=1e-3)
linear_model.fit(linear_X_train, flat_y_train_baligh)

linear_rows = []
for threshold in THRESHOLDS:
    predictions = predict_with_error_threshold(
        linear_model, linear_X_dev, float(threshold)
    )
    linear_rows.append(
        {
            "experiment": "linear_surface_v1",
            "threshold": round(float(threshold), 2),
            **binary_error_metrics(flat_dev_baligh, predictions),
        }
    )

linear_best = max(linear_rows, key=lambda row: (row["f1"], row["recall"]))

pd.DataFrame(linear_rows)

,experiment,threshold,tp,fp,fn,tn,precision,recall,f1,fp_per_1000
0,linear_surface_v1,0.05,10867,13036,995,28839,0.454629,0.916119,0.607689,242.588905
1,linear_surface_v1,0.10,9994,5813,1868,36062,0.632252,0.842522,0.722397,108.175000
2,linear_surface_v1,0.15,9445,3243,2417,38632,0.744404,0.796240,0.769450,60.349480
3,linear_surface_v1,0.20,9056,1958,2806,39917,0.822226,0.763446,0.791747,36.436720
4,linear_surface_v1,0.25,8750,1260,3112,40615,0.874126,0.737650,0.800110,23.447531
5,linear_surface_v1,0.30,8538,1007,3324,40868,0.894500,0.719777,0.797683,18.739416
6,linear_surface_v1,0.35,8343,807,3519,41068,0.911803,0.703338,0.794118,15.017586
7,linear_surface_v1,0.40,8132,656,3730,41219,0.925353,0.685550,0.787603,12.207604
8,linear_surface_v1,0.45,7851,511,4011,41364,0.938890,0.661861,0.776404,9.509277
9,linear_surface_v1,0.50,7545,420,4317,41455,0.947269,0.636065,0.761083,7.815844


In [9]:
# ====================================================
# E5: CRF (CONDITIONAL RANDOM FIELD) MULTICLASS MODEL
# ====================================================

crf_model = sklearn_crfsuite.CRF(
    algorithm="lbfgs",  # NOTE: lbfgs (https://en.wikipedia.org/wiki/Limited-memory_BFGS)
    # it's the default.
    c1=0.1,  # NOTE: L1 regularization
    c2=0.1,  # NOTE: L2 regularization
    max_iterations=100,
)
crf_model.fit(X_train_v1, y_train_baligh)

crf_rows = []
for threshold in THRESHOLDS:
    predictions = flatten(
        predict_crf_with_error_threshold(crf_model, X_dev_v1, float(threshold))
    )
    crf_rows.append(
        {
            "experiment": "crf_surface_v1",
            "threshold": round(float(threshold), 2),
            **binary_error_metrics(flat_dev_baligh, predictions),
        }
    )

crf_best = max(crf_rows, key=lambda row: (row["f1"], row["recall"]))

pd.DataFrame(crf_rows)

,experiment,threshold,tp,fp,fn,tn,precision,recall,f1,fp_per_1000
0,crf_surface_v1,0.05,10828,6418,1034,35457,0.627856,0.912831,0.743988,119.433537
1,crf_surface_v1,0.10,10442,3848,1420,38027,0.730721,0.880290,0.798562,71.608017
2,crf_surface_v1,0.15,10190,2661,1672,39214,0.792934,0.859046,0.824667,49.518953
3,crf_surface_v1,0.20,9998,1876,1864,39999,0.842008,0.842860,0.842433,34.910769
4,crf_surface_v1,0.25,9837,1469,2025,40406,0.870069,0.829287,0.849189,27.336844
5,crf_surface_v1,0.30,9703,1173,2159,40702,0.892148,0.817990,0.853461,21.828535
6,crf_surface_v1,0.35,9573,970,2289,40905,0.907996,0.807031,0.854541,18.050877
7,crf_surface_v1,0.40,9455,828,2407,41047,0.919479,0.797083,0.853917,15.408378
8,crf_surface_v1,0.45,9344,699,2518,41176,0.930399,0.787726,0.853139,13.007797
9,crf_surface_v1,0.50,9194,594,2668,41281,0.939313,0.775080,0.849330,11.053836


## Final models
Trying binary and trying with morph features


In [10]:
# =========================
# HELPERS for morph models
# =========================


def iter_sentences(path: Path):
    """Yield sentence-level token/category pairs from a split."""
    sentence = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            if sentence:
                yield sentence
                sentence = []
            continue
        token, label = line.split("\t")
        sentence.append((token, LABEL_TO_CATEGORY[label]))
    if sentence:
        yield sentence


def token_features(sent, index: int, with_extra_surface: bool = True):
    """Extract the surface feature set for one token."""
    token = sent[index][0]
    token_norm = normalize_token(token)
    row = {
        "bias": 1.0,  # NOTE: turned into sparse features
        f"token={token}": 1.0,
        f"norm={token_norm}": 1.0,
        f"shape={token_shape(token)}": 1.0,
        f"len={min(len(token), 20)}": 1.0,
        "is_digit": float(token.isdigit()),
        "is_punct": float(bool(PUNCT_RE.match(token))),
        "is_arabic": float(bool(token) and all(char in AR_CHARSET for char in token)),
    }

    for n in range(1, 5):
        row[f"prefix_{n}={token[:n]}"] = 1.0
        row[f"suffix_{n}={token[-n:]}"] = 1.0

    if index == 0:
        row["BOS"] = 1.0
    else:
        previous = sent[index - 1][0]
        row[f"prev_norm={normalize_token(previous)}"] = 1.0
        row["prev_is_punct"] = float(bool(PUNCT_RE.match(previous)))

    if index == len(sent) - 1:
        row["EOS"] = 1.0
    else:
        following = sent[index + 1][0]
        row[f"next_norm={normalize_token(following)}"] = 1.0
        row["next_is_punct"] = float(bool(PUNCT_RE.match(following)))

    if with_extra_surface:
        row[f"sent_len_bucket={min(len(sent) // 10, 10)}"] = 1.0
        row[f"position_bucket={min(int(10 * index / max(len(sent), 1)), 9)}"] = 1.0
        row["has_diacritic"] = float(
            any("\u064b" <= char <= "\u065f" for char in token)
        )
        row["has_repeated_char"] = float(
            any(a == b for a, b in zip(token, token[1:], strict=False))
        )
        row["starts_det"] = float(token_norm.startswith("ال"))
        padded = f"^{token_norm}$"
        for n in (2, 3, 4):
            for start in range(len(padded) - n + 1):
                row[f"char_{n}={padded[start : start + n]}"] = 1.0

    return row


MORPH_KEYS = (
    "pos",
    "ud",  # another kind of pos
    "lex",  # lexical structure
    "stem",
    "root",
    "pattern",
    "gen",
    "num",
    "per",
    "asp",
    "vox",  # voice
    "mod",
    "stt",  # diffentiveness
    "cas",
    "prc3",  # proclitics ادوات الوصل مثل الـ او فـ
    "prc2",
    "prc1",
    "prc0",
    "enc0",
    "source",
)


def compact_analysis(analysis: dict):
    """Keep only the morph fields used."""
    values = tuple(str(analysis.get(key, "NA")) for key in MORPH_KEYS)
    prob_buckets = []

    # NOTE: bucket log probs into ints
    for key in ("pos_logprob", "lex_logprob", "pos_lex_logprob"):
        value = float(analysis.get(key, -99.0))
        prob_buckets.append(-99 if value <= -90 else int(np.floor(value)))

    return (*values, *prob_buckets)


def add_morph_features(row: dict, analyses: list[tuple], index: int) -> None:
    """Add morphology-based features for one token from cache."""
    analysis = analyses[index]
    for key, value in zip(MORPH_KEYS, analysis[: len(MORPH_KEYS)], strict=True):
        # sparse features
        row[f"morph_{key}={value}"] = 1.0

    row["morph_is_backoff"] = float(analysis[MORPH_KEYS.index("source")] == "backoff")

    for key, bucket in zip(
        ("pos_logprob", "lex_logprob", "pos_lex_logprob"),
        analysis[len(MORPH_KEYS) :],
        strict=True,
    ):
        row[f"morph_{key}_bucket={bucket}"] = 1.0

    pos_index = MORPH_KEYS.index("pos")
    if index > 0:
        previous_pos = analyses[index - 1][pos_index]
        row[f"morph_prev_pos={previous_pos}"] = 1.0
    if index + 1 < len(analyses):
        next_pos = analyses[index + 1][pos_index]
        row[f"morph_next_pos={next_pos}"] = 1.0


def build_surface_v1_morph_features(sent, morph_cache):
    """Add morphology features on top of other features."""
    analyses = [morph_cache[token] for token, _ in sent]
    rows = []

    for index in range(len(sent)):
        row = surface_v1_token_features(sent, index).copy()
        add_morph_features(row, analyses, index)
        rows.append(row)

    return rows


def chunked(items, size: int):
    """Yield fixed-size chunks from an iterable."""
    iterator = iter(items)
    while chunk := list(islice(iterator, size)):
        yield chunk


def build_morph_cache(paths: list[Path]) -> dict[str, tuple]:
    """Run CAMeL disambiguation once and cache the compact morphology output."""
    unique_tokens = {
        token for path in paths for sent in iter_sentences(path) for token, _ in sent
    }

    disambiguator = MLEDisambiguator.pretrained()

    cache = {}
    for words in chunked(sorted(unique_tokens), 1000):
        results = disambiguator.disambiguate(words)
        for word, result in zip(words, results, strict=True):
            analysis = result.analyses[0].analysis if result.analyses else {}
            cache[word] = compact_analysis(analysis)

    return cache


def hash_sentence_batch(sentences, hasher, morph_cache):
    """Convert one sentence batch into hashed features and binary labels."""
    rows = []
    labels = []
    categories = []

    for sent in sentences:
        analyses = [morph_cache[token] for token, _ in sent] if morph_cache else None
        for index, (_, category) in enumerate(sent):
            row = token_features(sent, index)
            if analyses is not None:
                add_morph_features(row, analyses, index)
            rows.append(row)
            labels.append(category != NO_ERROR)
            categories.append(category)

    matrix = hasher.transform(rows).tocsr()
    return (
        matrix,
        np.asarray(labels, dtype=np.bool_),
        np.asarray(categories, dtype="U3"),
    )


def iter_hashed_batches(
    path: Path, hasher, morph_cache, batch_sentence_count: int = 250
):
    """Stream hashed feature batches from one split file."""
    sentence_batch = []
    for sent in iter_sentences(path):
        sentence_batch.append(sent)
        if len(sentence_batch) == batch_sentence_count:
            yield hash_sentence_batch(sentence_batch, hasher, morph_cache)
            sentence_batch = []
    if sentence_batch:
        yield hash_sentence_batch(sentence_batch, hasher, morph_cache)


def metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Compute binary metrics from boolean target arrays."""
    tp = int(np.sum(y_true & y_pred))
    fp = int(np.sum(~y_true & y_pred))
    fn = int(np.sum(y_true & ~y_pred))

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    return {
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "fp_per_1000": fp / len(y_true) * 1000,
        "tp": tp,
        "fp": fp,
        "fn": fn,
    }


def run_bin_model(with_morph: bool, shared_morph_cache=None):
    """Train the final binary classifier return the best dev run."""
    hasher = FeatureHasher(
        n_features=N_FEATURES,
        input_type="dict",
        alternate_sign=False,
        dtype=np.float32,
    )

    # NOTE: already built the morph cache earlier, reuse it to save time.
    morph_cache = shared_morph_cache if with_morph else None
    if with_morph and morph_cache is None:
        morph_cache = build_morph_cache([TRAIN_PATH, DEV_PATH])
    memory_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024

    models = {
        alpha: SGDClassifier(
            loss="log_loss",
            alpha=alpha,
            random_state=7,
            n_jobs=1,
            average=True,
        )
        for alpha in (3e-6, 1e-5, 3e-5)
    }

    classes = np.asarray([False, True])
    for _ in range(3):
        for matrix, batch_labels, _ in iter_hashed_batches(
            TRAIN_PATH, hasher, morph_cache
        ):
            for model in models.values():
                model.partial_fit(matrix, batch_labels, classes=classes)

    probabilities = {alpha: [] for alpha in models}
    dev_labels = []
    dev_categories = []

    for matrix, batch_labels, batch_categories in iter_hashed_batches(
        DEV_PATH, hasher, morph_cache
    ):
        for alpha, model in models.items():
            probabilities[alpha].append(model.predict_proba(matrix)[:, 1])
        dev_labels.append(batch_labels)
        dev_categories.append(batch_categories)

    y_dev = np.concatenate(dev_labels)
    dev_categories = np.concatenate(dev_categories)
    experiment_results = {}

    for alpha, _model in models.items():
        model_probabilities = np.concatenate(probabilities[alpha])
        rows = []
        for threshold in THRESHOLDS:
            row = metrics(y_dev, model_probabilities >= threshold)
            rows.append({"threshold": round(float(threshold), 2), **row})

        best = max(rows, key=lambda row: (row["f1"], row["recall"]))
        selected = model_probabilities >= best["threshold"]
        category_recall = {
            category: float(np.mean(selected[dev_categories == category]))
            for category in sorted(set(dev_categories) - {NO_ERROR})
        }
        experiment_results[alpha] = {
            "best_f1_row": best,
            "category_detection_recall": category_recall,
        }

    best_alpha, best_result = max(
        experiment_results.items(),
        key=lambda item: (
            item[1]["best_f1_row"]["f1"],
            item[1]["best_f1_row"]["recall"],
        ),
    )

    gc.collect()

    return {
        "alpha": best_alpha,
        "memory_mb": memory_mb,
        "best_row": best_result["best_f1_row"],
        "category_recall": best_result["category_detection_recall"],
    }

In [11]:
# ================================
# E6: LINEAR MULTICLASS + MORPH
# ================================

# NOTE:
shared_morph_cache = build_morph_cache([TRAIN_PATH, DEV_PATH])

X_train_v1_morph = [
    build_surface_v1_morph_features(sent, shared_morph_cache) for sent in train
]
X_dev_v1_morph = [
    build_surface_v1_morph_features(sent, shared_morph_cache) for sent in dev
]
flat_X_train_v1_morph = flatten(X_train_v1_morph)
flat_X_dev_v1_morph = flatten(X_dev_v1_morph)

linear_morph_vectorizer = DictVectorizer(sparse=True)
linear_morph_X_train = as_int32_csr(
    linear_morph_vectorizer.fit_transform(flat_X_train_v1_morph)
)
linear_morph_X_dev = as_int32_csr(
    linear_morph_vectorizer.transform(flat_X_dev_v1_morph)
)

linear_morph_model = SGDClassifier(
    loss="log_loss",
    random_state=7,
    max_iter=1000,
)
linear_morph_model.fit(linear_morph_X_train, flat_y_train_baligh)

linear_morph_rows = []
for threshold in THRESHOLDS:
    predictions = predict_with_error_threshold(
        linear_morph_model,
        linear_morph_X_dev,
        float(threshold),
    )
    linear_morph_rows.append(
        {
            "experiment": "linear_surface_v1_morph_v1",
            "threshold": round(float(threshold), 2),
            **binary_error_metrics(flat_dev_baligh, predictions),
        }
    )

linear_morph_best = max(linear_morph_rows, key=lambda row: (row["f1"], row["recall"]))

pd.DataFrame(linear_morph_rows)

,experiment,threshold,tp,fp,fn,tn,precision,recall,f1,fp_per_1000
0,linear_surface_v1_morph_v1,0.05,11151,8620,711,33255,0.564008,0.940061,0.705023,160.410890
1,linear_surface_v1_morph_v1,0.10,10719,4967,1143,36908,0.683348,0.903642,0.778205,92.431658
2,linear_surface_v1_morph_v1,0.15,10344,2852,1518,39023,0.783874,0.872028,0.825605,53.073301
3,linear_surface_v1_morph_v1,0.20,10066,1764,1796,40111,0.850888,0.848592,0.849738,32.826544
4,linear_surface_v1_morph_v1,0.25,9794,1222,2068,40653,0.889070,0.825662,0.856194,22.740384
5,linear_surface_v1_morph_v1,0.30,9631,985,2231,40890,0.907216,0.811920,0.856927,18.330015
6,linear_surface_v1_morph_v1,0.35,9452,806,2410,41069,0.921427,0.796830,0.854611,14.998976
7,linear_surface_v1_morph_v1,0.40,9227,657,2635,41218,0.933529,0.777862,0.848616,12.226213
8,linear_surface_v1_morph_v1,0.45,8958,541,2904,41334,0.943047,0.755185,0.838725,10.067551
9,linear_surface_v1_morph_v1,0.50,8663,444,3199,41431,0.951246,0.730315,0.826267,8.262463


In [12]:
# ============================
# E6B: CRF MULTICLASS + MORPH
# ============================

# NOTE: testing whether the CRF also benefits from the same morphology
# features that improved the final binary detector. it should but if the improvment
# is not worth it I will not update it.
crf_morph_model = sklearn_crfsuite.CRF(
    algorithm="lbfgs",
    c1=0.1,
    c2=0.1,
    max_iterations=100,
)
crf_morph_model.fit(X_train_v1_morph, y_train_baligh)

crf_morph_rows = []
for threshold in THRESHOLDS:
    predictions = flatten(
        predict_crf_with_error_threshold(
            crf_morph_model,
            X_dev_v1_morph,
            float(threshold),
        )
    )
    crf_morph_rows.append(
        {
            "experiment": "crf_surface_v1_morph_v1",
            "threshold": round(float(threshold), 2),
            **binary_error_metrics(flat_dev_baligh, predictions),
        }
    )

crf_morph_best = max(crf_morph_rows, key=lambda row: (row["f1"], row["recall"]))

pd.DataFrame(crf_morph_rows)

,experiment,threshold,tp,fp,fn,tn,precision,recall,f1,fp_per_1000
0,crf_surface_v1_morph_v1,0.05,10988,5615,874,36260,0.661808,0.926319,0.772036,104.490388
1,crf_surface_v1_morph_v1,0.10,10684,3559,1178,38316,0.750123,0.900691,0.818541,66.229972
2,crf_surface_v1_morph_v1,0.15,10495,2497,1367,39378,0.807805,0.884758,0.844532,46.467052
3,crf_surface_v1_morph_v1,0.20,10337,1877,1525,39998,0.846324,0.871438,0.858697,34.929378
4,crf_surface_v1_morph_v1,0.25,10195,1485,1667,40390,0.872860,0.859467,0.866112,27.634591
5,crf_surface_v1_morph_v1,0.30,10073,1216,1789,40659,0.892285,0.849182,0.870200,22.628729
6,crf_surface_v1_morph_v1,0.35,9973,1044,1889,40831,0.905237,0.840752,0.871804,19.427955
7,crf_surface_v1_morph_v1,0.40,9883,909,1979,40966,0.915771,0.833165,0.872517,16.915719
8,crf_surface_v1_morph_v1,0.45,9768,775,2094,41100,0.926492,0.823470,0.871948,14.422093
9,crf_surface_v1_morph_v1,0.50,9615,672,2247,41203,0.934675,0.810572,0.868211,12.505350


In [13]:
# ========================================
# E7: SURFACE-ONLY BINARY MODEL
# ========================================

# NOTE: This is the surface-only control for the final binary setup.
surface_result = run_bin_model(with_morph=False)

pd.DataFrame(
    [
        {
            "experiment": "surface_v2_binary",
            "alpha": surface_result["alpha"],
            "threshold": surface_result["best_row"]["threshold"],
            "f1": surface_result["best_row"]["f1"],
            "precision": surface_result["best_row"]["precision"],
            "recall": surface_result["best_row"]["recall"],
            "fp_per_1000": surface_result["best_row"]["fp_per_1000"],
            "memory_mb": round(surface_result["memory_mb"], 1),
        }
    ]
)

,experiment,alpha,threshold,f1,precision,recall,fp_per_1000,memory_mb
0,surface_v2_binary,0.000003,0.3,0.840676,0.929534,0.767324,12.840315,10748.3


In [14]:
# ========================================
# E8: SURFACE + MORPHOLOGY BINARY MODEL
# ========================================

# NOTE: The only intended difference from the previous cell is adding morphology.
morph_result = run_bin_model(with_morph=True, shared_morph_cache=shared_morph_cache)

pd.DataFrame(
    [
        {
            "experiment": "surface_v2_morph_v1_binary",
            "alpha": morph_result["alpha"],
            "threshold": morph_result["best_row"]["threshold"],
            "f1": morph_result["best_row"]["f1"],
            "precision": morph_result["best_row"]["precision"],
            "recall": morph_result["best_row"]["recall"],
            "fp_per_1000": morph_result["best_row"]["fp_per_1000"],
            "memory_mb": round(morph_result["memory_mb"], 1),
        }
    ]
)

,experiment,alpha,threshold,f1,precision,recall,fp_per_1000,memory_mb
0,surface_v2_morph_v1_binary,0.000003,0.3,0.872124,0.92561,0.824482,14.626793,10748.3


In [15]:
# ========================
# FINAL BINARY COMPARISON
# ========================

comparison = pd.DataFrame(
    [
        {
            "experiment": "surface_v2_binary",
            "alpha": surface_result["alpha"],
            "threshold": surface_result["best_row"]["threshold"],
            "f1": surface_result["best_row"]["f1"],
            "precision": surface_result["best_row"]["precision"],
            "recall": surface_result["best_row"]["recall"],
            "fp_per_1000": surface_result["best_row"]["fp_per_1000"],
            "memory_mb": round(surface_result["memory_mb"], 1),
        },
        {
            "experiment": "surface_v2_morph_v1_binary",
            "alpha": morph_result["alpha"],
            "threshold": morph_result["best_row"]["threshold"],
            "f1": morph_result["best_row"]["f1"],
            "precision": morph_result["best_row"]["precision"],
            "recall": morph_result["best_row"]["recall"],
            "fp_per_1000": morph_result["best_row"]["fp_per_1000"],
            "memory_mb": round(morph_result["memory_mb"], 1),
        },
    ]
)

comparison

,experiment,alpha,threshold,f1,precision,recall,fp_per_1000,memory_mb
0,surface_v2_binary,0.000003,0.3,0.840676,0.929534,0.767324,12.840315,10748.3
1,surface_v2_morph_v1_binary,0.000003,0.3,0.872124,0.925610,0.824482,14.626793,10748.3


In [16]:
category_recall = pd.DataFrame(
    {
        "surface_v2_binary": surface_result["category_recall"],
        "surface_v2_morph_v1_binary": morph_result["category_recall"],
    }
).round(4)

category_recall

,surface_v2_binary,surface_v2_morph_v1_binary
MG,0.8863,0.9086
MO,0.1860,0.2093
OT,0.8512,0.9248
PC,0.5000,0.4900
SP,0.8727,0.9583
SY,0.5951,0.6518
UNK,0.3600,0.3937


## Findings


In [17]:
experiment_summary = pd.DataFrame(
    [
        {
            "experiment": "all_uc_baseline",
            "f1": all_uc_row["f1"],
            "precision": all_uc_row["precision"],
            "recall": all_uc_row["recall"],
            "fp_per_1000": all_uc_row["fp_per_1000"],
            "note": "predicting no errors",
        },
        {
            "experiment": "token_memory_best_f1",
            "f1": token_memory_best["f1"],
            "precision": token_memory_best["precision"],
            "recall": token_memory_best["recall"],
            "fp_per_1000": token_memory_best["fp_per_1000"],
            "note": "too many false positives and that's makes sense",
        },
        {
            "experiment": "calibrated_token_memory_best_f1",
            "f1": calibrated_memory_best["f1"],
            "precision": calibrated_memory_best["precision"],
            "recall": calibrated_memory_best["recall"],
            "fp_per_1000": calibrated_memory_best["fp_per_1000"],
            "note": "better but lower recall than linear and crf",
        },
        {
            "experiment": "linear_surface_v1",
            "f1": linear_best["f1"],
            "precision": linear_best["precision"],
            "recall": linear_best["recall"],
            "fp_per_1000": linear_best["fp_per_1000"],
            "note": "best multiclass baseline",
        },
        {
            "experiment": "linear_surface_v1_morph_v1",
            "f1": linear_morph_best["f1"],
            "precision": linear_morph_best["precision"],
            "recall": linear_morph_best["recall"],
            "fp_per_1000": linear_morph_best["fp_per_1000"],
            "note": "multiclass linear model with morphology",
        },
        {
            "experiment": "crf_surface_v1",
            "f1": crf_best["f1"],
            "precision": crf_best["precision"],
            "recall": crf_best["recall"],
            "fp_per_1000": crf_best["fp_per_1000"],
            "note": "selected deployment model; direct tags without morphology",
        },
        {
            "experiment": "crf_surface_v1_morph_v1",
            "f1": crf_morph_best["f1"],
            "precision": crf_morph_best["precision"],
            "recall": crf_morph_best["recall"],
            "fp_per_1000": crf_morph_best["fp_per_1000"],
            "note": "best dev metrics; optional morphology upgrade",
        },
        {
            "experiment": "surface_v2_binary",
            "f1": surface_result["best_row"]["f1"],
            "precision": surface_result["best_row"]["precision"],
            "recall": surface_result["best_row"]["recall"],
            "fp_per_1000": surface_result["best_row"]["fp_per_1000"],
            "note": "best surface-only binary run",
        },
        {
            "experiment": "surface_v2_morph_v1_binary",
            "f1": morph_result["best_row"]["f1"],
            "precision": morph_result["best_row"]["precision"],
            "recall": morph_result["best_row"]["recall"],
            "fp_per_1000": morph_result["best_row"]["fp_per_1000"],
            "note": "best binary comparison; does not directly provide tags",
        },
    ]
).round(4)

experiment_summary

,experiment,f1,precision,recall,fp_per_1000,note
0,all_uc_baseline,0.0000,0.0000,0.0000,0.0000,predicting no errors
1,token_memory_best_f1,0.4167,0.3226,0.5880,272.4938,too many false positives and that's makes sense
2,calibrated_token_memory_best_f1,0.7683,0.9434,0.6480,8.5788,better but lower recall than linear and crf
3,linear_surface_v1,0.8001,0.8741,0.7376,23.4475,best multiclass baseline
4,linear_surface_v1_morph_v1,0.8569,0.9072,0.8119,18.3300,multiclass linear model with morphology
5,crf_surface_v1,0.8545,0.9080,0.8070,18.0509,selected deployment model; direct tags without...
6,crf_surface_v1_morph_v1,0.8725,0.9158,0.8332,16.9157,best dev metrics; optional morphology upgrade
7,surface_v2_binary,0.8407,0.9295,0.7673,12.8403,best surface-only binary run
8,surface_v2_morph_v1_binary,0.8721,0.9256,0.8245,14.6268,best binary comparison; does not directly prov...


## Deployment model choice


In [18]:
# ========================================
# DEPLOYMENT MODEL SELECTION
# ========================================

# The surface-only CRF is the current deployment choice because it predicts
# Baligh tags directly without adding CAMeL morphology latency at inference.
deployment_model_choice = {
    "role": "deployment_final",
    "model": "crf_surface_v1",
    "why": "direct multiclass tags with surface-only inference",
    "threshold": crf_best["threshold"],
    "f1": crf_best["f1"],
    "precision": crf_best["precision"],
    "recall": crf_best["recall"],
    "fp_per_1000": crf_best["fp_per_1000"],
}

# Keep the morphology CRF documented as the research-best upgrade candidate.
research_best_model_choice = {
    "role": "research_best",
    "model": "crf_surface_v1_morph_v1",
    "why": "highest dev F1, but requires morphology at inference",
    "threshold": crf_morph_best["threshold"],
    "f1": crf_morph_best["f1"],
    "precision": crf_morph_best["precision"],
    "recall": crf_morph_best["recall"],
    "fp_per_1000": crf_morph_best["fp_per_1000"],
}

final_model_choices = pd.DataFrame(
    [deployment_model_choice, research_best_model_choice]
).round(4)

final_model_choices

,role,model,why,threshold,f1,precision,recall,fp_per_1000
0,deployment_final,crf_surface_v1,direct multiclass tags with surface-only infer...,0.35,0.8545,0.9080,0.8070,18.0509
1,research_best,crf_surface_v1_morph_v1,"highest dev F1, but requires morphology at inf...",0.40,0.8725,0.9158,0.8332,16.9157
